In [1]:
from typing import Any
import torch
import torch.nn as nn
import torch.nn.functional as f
from torchvision.models import resnet50, swin_t, swin_s, swin_b, swin_v2_t, swin_v2_s, swin_v2_b
from torchvision.models import ResNet50_Weights, Swin_T_Weights, Swin_S_Weights, Swin_B_Weights, Swin_V2_T_Weights, Swin_V2_S_Weights, Swin_V2_B_Weights
from enum import Enum
from vision_studio.models import BaseModel, InputSpec, OutputSpec
from vision_studio.types import ClassificationPostprocessOutput, LossOutput
from vision_studio.evaluate import ClassificationEvaluator
from vision_studio.dataset import ImageNetClassificationDataset
from vision_studio.data_loader import SimpleDataLoader
from vision_studio.trainer import WandbTrainer
from vision_studio.augmentation import Resize
from torch.optim import SGD, AdamW

In [2]:
class ResNet50(BaseModel):
    def __init__(self, num_classes: int, weights: ResNet50_Weights | None):
        super().__init__()
        self.num_classes = num_classes

        self.model = resnet50(weights=weights)

        model_in_features = self.model.fc.in_features
        self.model.fc = nn.Linear(model_in_features, num_classes)

    @property
    def input_spec(self) -> InputSpec:
        """Expected input specification."""
        return InputSpec(dtype=torch.float32)

    @property
    def output_spec(self) -> OutputSpec:
        """Output specification for forward()."""
        return OutputSpec(dtype=torch.float32)

    def forward(self, inputs):
        """Return ONLY logits for speed."""
        logits = self.model(inputs)
        return logits

    def postprocess(self, logits) -> ClassificationPostprocessOutput:
        """Task-specific postprocessing for classification."""
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)

        return ClassificationPostprocessOutput(
            logits=logits,
            probs=probs,
            labels=labels,
        )

    def compute_loss(self, logits, targets) -> LossOutput:
        """Compute loss from raw logits."""
        labels = targets["label"]
        loss = f.cross_entropy(logits, labels)
        return LossOutput(loss=loss)

In [3]:


class SwinModelSelect(Enum):
    SWIN_T = 'SWIN_T'
    SWIN_S = 'SWIN_S'
    SWIN_B = 'SWIN_B'
    SWIN_V2_T = 'SWIN_V2_T'
    SWIN_V2_S = 'SWIN_V2_S'
    SWIN_V2_B = 'SWIN_V2_B'

class SwinTransformer(BaseModel):
    def __init__(self, model_select: SwinModelSelect, num_classes: int, weights: Any):
        super().__init__()
        self.num_classes = num_classes

        if model_select == SwinModelSelect.SWIN_T:
            if weights is not None and not isinstance(weights, Swin_T_Weights):
                raise ValueError(f"Expected weights to be of type Swin_T_Weights, got {type(weights)}")
            self.model = swin_t(weights=weights)
        elif model_select == SwinModelSelect.SWIN_S:
            if weights is not None and not isinstance(weights, Swin_S_Weights):
                raise ValueError(f"Expected weights to be of type Swin_S_Weights, got {type(weights)}")
            self.model = swin_s(weights=weights)
        elif model_select == SwinModelSelect.SWIN_B:
            if weights is not None and not isinstance(weights, Swin_B_Weights):
                raise ValueError(f"Expected weights to be of type Swin_B_Weights, got {type(weights)}")
            self.model = swin_b(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_T:
            if weights is not None and not isinstance(weights, Swin_V2_T_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_T_Weights, got {type(weights)}")
            self.model = swin_v2_t(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_S:
            if weights is not None and not isinstance(weights, Swin_V2_S_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_S_Weights, got {type(weights)}")
            self.model = swin_v2_s(weights=weights)
        elif model_select == SwinModelSelect.SWIN_V2_B:
            if weights is not None and not isinstance(weights, Swin_V2_B_Weights):
                raise ValueError(f"Expected weights to be of type Swin_V2_B_Weights, got {type(weights)}")
            self.model = swin_v2_b(weights=weights)

        incoming_signal_size = self.model.head.in_features
        self.model.head = nn.Linear(incoming_signal_size, num_classes)

    @property
    def input_spec(self) -> InputSpec:
        """Expected input specification."""
        return InputSpec(dtype=torch.float32)

    @property
    def output_spec(self) -> OutputSpec:
        """Output specification for forward()."""
        return OutputSpec(dtype=torch.float32)

    def forward(self, inputs):
        """Return ONLY logits for speed."""
        _, _, h, w = inputs.shape

        if h % 4 != 0 or w % 4 != 0:
            raise ValueError(f"Input size must be divisible by 4, got {h}x{w}")
        logits = self.model(inputs)
        return logits

    def postprocess(self, logits) -> ClassificationPostprocessOutput:
        """Task-specific postprocessing for classification."""
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(logits, dim=1)

        return ClassificationPostprocessOutput(
            logits=logits,
            probs=probs,
            labels=labels,
        )

    def compute_loss(self, logits, targets) -> LossOutput:
        """Compute loss from raw logits."""
        labels = targets["label"]
        loss = f.cross_entropy(logits, labels)
        return LossOutput(loss=loss)

In [4]:
resize = Resize(224, 224)

dataset_train = ImageNetClassificationDataset(root_dir="/home/bullseye/datasets/gtsrb_out", transform=resize)
dataset_val = ImageNetClassificationDataset(root_dir="/home/bullseye/datasets/gtsrb_out", split="val", transform=resize)
print("Number of training samples:", len(dataset_train))
print("Number of validation samples:", len(dataset_val))

Number of training samples: 35347
Number of validation samples: 12630


In [ ]:
data_loader_train = SimpleDataLoader(dataset=dataset_train, dataset_percentage_per_epoch=10, batch_size=96, shuffle=True)
data_loader_val = SimpleDataLoader(dataset=dataset_val, batch_size=96, shuffle=False)

In [6]:
model = ResNet50(num_classes=dataset_train.get_num_classes(), weights=ResNet50_Weights.IMAGENET1K_V2)
adamW_optimizer = AdamW(model.parameters(), lr=0.001)
sgd_optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# Create evaluator with type-safe metrics
evaluator = ClassificationEvaluator(
    num_classes=dataset_train.get_num_classes(),
    topk=(1, 5)
)

# Create trainer with evaluator
trainer = WandbTrainer(
    optimizer=adamW_optimizer,
    evaluator=evaluator,
    epochs=10,
    project="cv-course",
    run_name="resnet50-classification",
    device="cuda"
)

In [7]:
trainer.fit(model=model, train_loader=data_loader_train, val_loader=data_loader_val)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/bullseye/.netrc.
wandb: Currently logged in as: moritz-zideck (moritz-zideck-tu-wien) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/10


100%|██████████| 369/369 [02:19<00:00,  2.65it/s]


Epoch 2/10


100%|██████████| 369/369 [02:20<00:00,  2.63it/s]


...

Epoch 7/10


 21%|██▏       | 79/369 [00:30<01:52,  2.58it/s]


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x733b1dab40e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 733b1dab7650, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 733b1dab7530, raw_cell="trainer.fit(model=model, train_loader=data_loader_.." transformed_cell="trainer.fit(model=model, train_loader=data_loader_.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2Bbullseye/home/bullseye/programming/CV_VisionStudio/notebooks/cv-course.ipynb#W6sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost